In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from ydata_profiling import ProfileReport
import pickle
import random
import itertools

pd.options.display.max_columns = None
pd.options.display.max_colwidth = 50

data_folder = "../data/"
images_folder = "../data/images/"
pipelines_folder = "../pipelines/"
df_total = pd.read_csv(os.path.join(data_folder, 'items_phase_1.csv'))
df_train = pd.read_csv(os.path.join(data_folder, 'items_train.csv'))
df_task_1 = pd.read_csv(os.path.join(data_folder, 'task_1.csv'))

# Notebook `items_phase_1.csv`

In [2]:
# profile = ProfileReport(df_total, title="items_phase_1", explorative=True)
# profile.to_notebook_iframe()

In [3]:
print("Length of dataset:", len(df_total))

Length of dataset: 199835


In [4]:
df_total.sample(2)

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo
126101,991939,91.90,783,['1'],NaN,Θήκη πιστωτικών καρτών LAUREN RALPH LAUREN,Θήκη πιστωτικών καρτών Lauren Ralph Lauren Zip...,gr
147231,977989,45.95,230,['1'],NaN,Pasni pas Tommy Hilfiger,Tommy Hilfiger Pasni pas Turnlock 2.5 AW0AW147...,si


## Key takeaway
- missingy se musi resit u:
    - description (3%)
    - brandEditionTagId (99.8%) - to je asi target?
    - colorTagIdsString (3.1%)
---
# Notebook `task_1.csv`
- kazdy radek je jedna skupina se sloupci item - item4(to jsou id do items_phase_1.csv - itemId)
- Kazdy 

In [5]:
print("Length of dataset:", len(df_task_1))

Length of dataset: 15000


In [6]:
# profile = ProfileReport(df_task_1, title="task_1", explorative=True)
# profile.to_notebook_iframe()

In [7]:
df_task_1.head()

,item1,item2,item3,item4,item5
0,130622,292253,463442,483968,1253745
1,82627,388496,553738,638400,884327
2,46130,333489,644448,848154,1178149
3,150796,248537,742067,1206230,1280786
4,76610,196894,345145,620255,932761


---
# Dataset `items_train.csv`
- obsahuje toho min, vim ze ma stejny hodnoty v departmentIds


In [8]:
print("Total records:", len(df_train))
print("Total null values:\n", df_train.isnull().sum())

Total records: 928234
Total null values:
 itemId                    0
price                     0
colorTagIdsString     27834
departmentIds             0
brandEditionTagId    925518
title                     0
description           35473
geo                       0
label                     0
dtype: int64


In [9]:
df_train.sample(2)

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo,label
380102,448620,20.14,"6455,1443396",['11'],NaN,Béžové chino nohavice Celio Arobert,Model: Arobert\nFarba: béžová\nStrih: chino\nD...,sk,102876
373911,53490,59.90,780,['1'],NaN,Βραχιόλι HUGO,Βραχιόλι HUGO 50544514 Ασημί,gr,58181


In [10]:
# profile = ProfileReport(df_train, title="task_1", explorative=True)
# profile.to_notebook_iframe()


---
# Vycisteni dat 
- prevod na spolecnou menu 
- normalizace meny v danem geo uzemi
- doplneni null hodnot
- null ve sloupci `colorTagIdString` muzu nahradit 0 
- `colorTagIdString` a `departmentIds` je potreba roztrhnout - obsahuji vice hodnot oddelenych carkou

## Tvorba preprocessing pipeline

In [11]:
df_total[df_total["brandEditionTagId"] == 0]

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo


In [12]:
df_train.columns

Index(['itemId', 'price', 'colorTagIdsString', 'departmentIds',
       'brandEditionTagId', 'title', 'description', 'geo', 'label'],
      dtype='object')

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from PriceGeoTransformer import PriceGeoTransformer
from DepartmentIdsTransformer import DepartmentIdsCleaner
import numpy as np

imput_zero_cols = ['colorTagIdsString', "brandEditionTagId"]
input_zero_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
])

input_unknown_cols = ["description"]
input_unknown_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value="Unknown")),
])



# prevadi na stejny format jako jsou barvy - cisla oddelena carkou 
department_features = ['departmentIds']
department_transformer = Pipeline(steps=[
    ('DepartmentIdsCleaner', DepartmentIdsCleaner())
])

# impute missing geo and convert back to pandas
imputer_step = ColumnTransformer(
    transformers=[
        ('geo_imp', SimpleImputer(strategy='constant', fill_value='<UNK>'), ['geo']),
        ('price_imp', SimpleImputer(strategy='median'), ['price'])
    ], 
    remainder='passthrough',
    verbose_feature_names_out=False 
).set_output(transform="pandas")

categorical_features = ['geo',"price"]
categorical_transformer = Pipeline(steps=[
    ("imputer", imputer_step),
    ('PriceGeoTransformer', PriceGeoTransformer())
])


# Combine preprocessing for numeric and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('zero', input_zero_transformer, imput_zero_cols),
        ('unknown', input_unknown_transformer, input_unknown_cols),
        ('geo', categorical_transformer, categorical_features),
        ('department', department_transformer, department_features)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

preprocessor.set_output(transform="pandas")



pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

X_train = df_train.drop(columns=['label'])
y_train = df_train['label']

X_train_transformed = pipeline.fit_transform(X_train)
df_train_transformed = X_train_transformed.copy()
df_train_transformed['label'] = y_train.values
df_total_transformed = pipeline.transform(df_total)

In [25]:
pickle.dump(pipeline, open(os.path.join(pipelines_folder, 'preprocessing_pipeline.pkl'), 'wb'))

## Příprava pro PyTorch dataset - Vocabulary pro transformaci kategorií
- PyTorch bude použit na vytvoření embeddingů sloupců s více kategorijema a na kategorické sloupce
- PyTorch umí totiž lépe tvořit řídké matice
- Vytvoříme si mappingy pro jednotlivé kategorie...

In [31]:
from GlamiDatasetVocabulary import GlamiVocabularyManager
vocab_manager = GlamiVocabularyManager() # vytvori vsechny potrebne mappingy kategorii do ciselnych hodnot
vocab_manager.fit(df_total_transformed)
vocab_manager.save()

## PyTorch Dataset


In [32]:
from GlamiItemDataset import GlamiItemDataset
dat = GlamiItemDataset(df_train_transformed, vocab_manager, images_folder, tokenizer_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", max_len=128)

In [33]:
df_total_transformed.iloc[0]

colorTagIdsString                                              771,772
brandEditionTagId                                                    0
description          Πορτοφόλι της κολεξιόν Tommy Hilfiger. Μοντέλο...
price_eur                                                         69.9
price_scaled                                                 -0.249997
geo                                                                 gr
departmentIds                                                       11
itemId                                                          451136
title                Δερμάτινη θήκη για κάρτες Tommy Hilfiger χρώμα...
Name: 0, dtype: object

In [34]:
dat[0]

{'item_id': '692210',
 'price': tensor([-0.2543]),
 'geo': tensor(1),
 'colors': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0.]),
 'departments': tensor([0., 1., 0.]),
 'input_ids': tensor([     0,   9504,   7405,  82507,      7,    616, 104519,  72904,    812,
            992,  69723,     19,      2,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,    

## V tomto stavu:
- Nejsou null hodnoty ve sloupcích:
    - colorTagIdString
    - description
    -  brandEditionTagId
- Barvy a deptId maji stejny format - cisla oddelena carkou
- chybi udelat OneHot encodingy kategorickejch - to udelame v PyTorchi
--- 
## Predpocitani embeddingu obrazku
- pro rychlejsi trenovani pouzijeme predtrenovany model a tim spocteme vsechny embeddingy pro obrazky 


## Tvorba siamese datasetu
- dataset obsahuje páry z originálního datasetu teré mají jako label 1 pokud mají stejné labely uvitř páru, jinak 0
- create_balanced_pairs jich vytvori tolik aby byly vyvazene labely 

In [35]:
def create_balanced_pairs(df, item_col='itemId', group_col='itemGroupId'):
    """
    Vygeneruje vyvážený dataset párů (50 % pozitivních, 50 % negativních).
    """
    print("Generuji pozitivní páry (duplikáty)...")
    positive_pairs = []
    
    # 1. Najdeme všechny produkty, které patří k sobě
    for _, group_df in df.groupby(group_col):
        items = group_df[item_col].tolist()
        if len(items) >= 2:
            # Vytvoříme všechny možné dvojice v rámci stejné skupiny
            positive_pairs.extend(list(itertools.combinations(items, 2)))
            
    num_positives = len(positive_pairs)
    print(f"Nalezeno {num_positives} pozitivních párů.")
    
    # 2. Rychlý slovník pro kontrolu negativních párů
    item_to_group = dict(zip(df[item_col], df[group_col]))
    all_items = df[item_col].tolist()
    
    print("Generuji negativní páry (odlišné produkty)...")
    negative_pairs = []
    
    # 3. Náhodně losujeme dvojice, dokud jich nemáme stejně jako pozitivních
    while len(negative_pairs) < num_positives:
        item1 = random.choice(all_items)
        item2 = random.choice(all_items)
        
        # Musí to být různé produkty a nesmí patřit do stejné skupiny!
        if item1 != item2 and item_to_group[item1] != item_to_group[item2]:
            negative_pairs.append((item1, item2))
            
    pos_df = pd.DataFrame(positive_pairs, columns=['item_id_1', 'item_id_2'])
    pos_df['label'] = 1.0
    
    neg_df = pd.DataFrame(negative_pairs, columns=['item_id_1', 'item_id_2'])
    neg_df['label'] = 0.0
    
    pairs_df = pd.concat([pos_df, neg_df]).sample(frac=1.0, random_state=42).reset_index(drop=True)
    
    print(f"Hotovo! Výsledný dataset má {len(pairs_df)} párů.")
    return pairs_df

In [36]:
from GlamiSiameseDataset import GlamiSiameseDataset
my_pairs_df = create_balanced_pairs(df_train_transformed)

# 2. Vytvoříš "knihovnu" (To je tvůj včerejší kód)
base_dataset = GlamiItemDataset(df=df_train_transformed, vocab_manager=vocab_manager, images_dir="cesta", ...)

# 3. Vytvoříš finální Siamský dataset připravený pro trénink
siamese_dataset = GlamiSiameseDataset(item_dataset=base_dataset, pairs_df=my_pairs_df)

# Zkušební tisk:
vzorek = siamese_dataset[0]
print("Label vzorku:", vzorek['label'].item())
print("Cena prvního produktu:", vzorek['item1']['price'].item())
print("Cena druhého produktu:", vzorek['item2']['price'].item())

SyntaxError: positional argument follows keyword argument (2942308897.py, line 5)